# 01 - Train and monitor

Trains in the background with **Pause / Resume / Stop** buttons and a live dashboard, then scores the held-out
TEST block against baselines and analyses every head per horizon. The notebook stays responsive during training
(the buttons only work while no cell is running). Stop ends training after the current batch; the run is still
evaluated, calibrated and saved.

The weights that are evaluated, calibrated and saved are the best-validation epoch's (the *served* epoch), also
when training ran all `EPOCHS`. Every verdict below is graded against noise: consecutive 1-minute samples share
most of their target bars, so the chance bands count fewer effective samples than bars (about `samples // bars ahead`). Horizons keep
their colours in every figure (h0 blue, h1 orange, h2 green); solid lines are validation, dotted lines training.

In [1]:
# Parameters
CONFIG_PATH = "../configs/default.yaml"
CSV_PATH = "../binance_btcusdt_1min_ccxt.csv"
RUNS_DIR = "../runs"
OVERRIDES = {}            # e.g. {"EPOCHS": 5, "LR": 5e-4, "FOLD_INDEX": -2, "BATCH_SIZE": 64}
EPOCHS = None             # None -> Config.EPOCHS
CALIBRATE_LOSS_WEIGHTS = True

In [2]:
from IPython.display import HTML, Markdown, display

import neural_trade  # first: on Windows it puts the CUDA DLLs on PATH before TensorFlow loads
from neural_trade.core.config import Config
from neural_trade.data.processor import split_arrays
from neural_trade.evaluation.baselines import BaselineSet
from neural_trade.evaluation.frame import PredictionFrame
from neural_trade.evaluation.report import evaluate
from neural_trade.experiments.run_context import RunContext
from neural_trade.notebook import TrainingSession
from neural_trade.registries.visualizations import Visualizations
from neural_trade.strategy import var_scale_from
from neural_trade.visualization import analytics_tables as AT
from neural_trade.visualization.indicator_evolution import applied_periods, indicator_applied_periods, indicator_summary
import tensorflow as tf

print("GPU:", tf.config.list_physical_devices("GPU") or "none - training will run on the CPU")
cfg = Config.from_yaml(CONFIG_PATH).override(CSV_PATH=CSV_PATH, **OVERRIDES)
ctx = RunContext.create(cfg, root=RUNS_DIR, tags=["notebook"])
print("run:", ctx.run_dir)

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
run: ..\runs\20260924T182915Z-1aeff1c-dirty-af67ee43


## Train

This cell returns immediately; the dashboard below keeps updating. The next cell waits for the run to finish.

In [3]:
session = TrainingSession(ctx.config, run_context=ctx, epochs=EPOCHS, calibrate=CALIBRATE_LOSS_WEIGHTS)
display(session.widget())
session.start()

In [4]:
result = session.wait()   # blocks until training, evaluation and calibration are done
print(session.status, "-", len(session.history), "epochs; served weights: epoch", result.weights_epoch,
      f"(val loss {result.weights_val_loss:.4f}, {result.weights_source})")

finished - 20 epochs; served weights: epoch 19 (val loss 4.8875, best validation epoch, restored after training (EarlyStopping did not stop the run: 20 epochs ran))


### Training record

A static copy of the live dashboard, readable in the saved notebook without a running kernel. The health tiles
(hover one for what it checks) and the epoch table come first; then every per-epoch metric: total loss with the
best and served epochs, the validation loss by term as each term enters the total, MCC and balanced accuracy of
the direction head and the price head with 95% chance bands, Brier skill against the base rate, ECE and PIT-KS
with the level a calibrated head stays under, the up-call bias, the physics terms, learning rates and gradient
norm. Then the per-horizon direction metrics against their no-skill references, the loss terms per horizon, and
the loss batch by batch.

In [5]:
display(HTML(session.health_html()))
session.curves_figure().show()
session.direction_figure().show()
session.loss_terms_figure().show()
session.batch_figure().show()

term,weight in the total,epoch 19 (served): val,share of val,train,epoch 20 (last): val
point,λ 1.86 / 0.717 / 0.684 (inside),0.4434,9.1%,0.9717,0.4679
trend,λ_ext 0.1 (inside) × 1,0.03906,0.8%,0.09656,0.04157
direction BCE,λ_dir 0.659 × 1,1.385,28.3%,1.354,1.386
NLL,λ_var 0.369 × 1,0.9672,19.8%,1.444,0.9736
CRPS,λ_crps 0.775,0.7596,15.5%,1.189,0.7764
soft ECE,λ_soft_ece 2.24,0.7688,15.7%,0.3214,0.7789
volatility,λ_vol 3.32 (inside) × 0.1,0.1414,2.9%,0.1147,0.1371
physics,λ 0.1 (inside),0.2346,4.8%,0.1223,0.2324
T-perp,,0.2039,,0.02071,0.2009
Casimir,,4.7e-05,,1.7e-05,4.143e-05


## Evaluate on the TEST block

Baselines are fitted on the train block; the confidence threshold and calibration come from the cal block. The
report scores the served heads and the raw price heads (before delta shrinkage) side by side; the baseline table
gives the model and baseline values, the margin, a noise test (HAC Diebold-Mariano or block bootstrap) and the
verdict.

In [6]:
blocks = split_arrays(ctx.config)
baselines = BaselineSet.fit(blocks["train"]["X"], blocks["train"]["y"], blocks["train"]["last_close"],
                            ctx.config.DIR_DEADBAND_BPS)
test = PredictionFrame.from_result(result, "test")
cal = PredictionFrame.from_result(result, "cal")
raw_test = result.predictions["delta"]
betas = result.calibration_pipeline.delta_scale if result.calibration_pipeline is not None else None
report = evaluate(test, ctx.config, baselines=baselines, cal_frame=cal, run_id=ctx.run_id,
                  raw_delta=raw_test, delta_scale=betas)
report.to_json(ctx.path("eval_report_test.json"))
display(Markdown(report.to_markdown(ctx.path("eval_report_test.md"))))
display(AT.styled(AT.baseline_table(report)))

Dataset length after cleaning: 43500


Date range after cleaning: 2025-10-11 02:30:00+00:00 to 2025-11-10 07:29:00+00:00


# Evaluation report - test split - run `20260924T182915Z-1aeff1c-dirty-af67ee43`

n = 7236 samples, one per 1-minute bar. Direction metrics count only moves beyond 5 bps (the neutral mask). Consecutive samples share most of their target window, so n_eff = n // bars ahead counts the non-overlapping outcomes.

## Direction heads: P(up) > 0.5 on moves beyond 5 bps

Up is the positive class. Temperature scaling does not move P(up) across 0.5, so the counts and rates are the same before and after calibration; Brier and ECE are not.

| metric | h0 (10 bars) | h1 (15 bars) | h2 (20 bars) |
|---|---|---|---|
| n scored (outside the deadband) | 5345 | 5697 | 5874 |
| n_eff of the scored moves (n scored // bars ahead) | 534 | 379 | 293 |
| true up-rate | 0.5139 | 0.5147 | 0.5189 |
| calls up (predicted up-rate) | 0.4958 | 0.4308 | 0.4848 |
| accuracy | 0.4849 | 0.4811 | 0.5026 |
| balanced accuracy | 0.4850 | 0.4831 | 0.5031 |
| precision (up) | 0.4989 | 0.4951 | 0.5221 |
| recall / sensitivity (up) | 0.4813 | 0.4144 | 0.4879 |
| specificity (down) | 0.4888 | 0.5519 | 0.5184 |
| F1 (up) | 0.4899 | 0.4512 | 0.5044 |
| MCC | -0.0299 | -0.0340 | 0.0063 |
| AUC | 0.4855 | 0.4781 | 0.5044 |
| Brier | 0.2519 | 0.2529 | 0.2507 |
| ECE (positive class) | 0.0391 | 0.0477 | 0.0226 |
| ECE of a constant 0.5 (= distance of the up-rate from 0.5) | 0.0139 | 0.0147 | 0.0189 |
| TP / FP / TN / FN | 1322 / 1328 / 1270 / 1425 | 1215 / 1239 / 1526 / 1717 | 1487 / 1361 / 1465 / 1561 |
| Gaussian readout: calls up | 0.3886 | n/a (beta = 0: readout is the constant 0.5) | 0.5410 |
| Gaussian readout: MCC | -0.0034 | n/a (beta = 0: readout is the constant 0.5) | -0.0185 |
| Gaussian readout: AUC | 0.5020 | n/a (beta = 0: readout is the constant 0.5) | 0.4826 |
| Gaussian readout: Brier | 0.2501 | n/a (beta = 0: readout is the constant 0.5) | 0.2502 |
| Gaussian readout: ECE | 0.0150 | n/a (beta = 0: readout is the constant 0.5) | 0.0187 |
| Gaussian readout of the raw heads: calls up | 0.3886 | 0.4867 | 0.5410 |
| Gaussian readout of the raw heads: MCC | -0.0034 | -0.0099 | -0.0185 |
| Gaussian readout of the raw heads: AUC | 0.5020 | 0.4805 | 0.4826 |
| Gaussian readout of the raw heads: Brier | 0.2518 | 0.2587 | 0.2587 |
| Gaussian readout of the raw heads: ECE | 0.0325 | 0.0552 | 0.0693 |

beta = 0 for h1: the served delta is 0 there, so its Gaussian readout P(up | the move leaves the deadband) is the constant 0.5: it calls up on no bar and has AUC 0.5 and Brier 0.25 by construction, so those cells are n/a. The rows "Gaussian readout of the raw heads" score the raw price heads' readout (with the served variance).

## Price heads (dollars)

| metric | h0 (10 bars) | h1 (15 bars) | h2 (20 bars) |
|---|---|---|---|
| RMSE ($), served | 196.23 | 236.10 | 269.23 |
| RMSE ($), raw heads | 197.25 | 243.27 | 275.70 |
| RMSE ($), zero prediction | 196.21 | 236.10 | 269.11 |
| MAE ($), served | 145.68 | 175.66 | 200.03 |
| MAE ($), raw heads | 146.68 | 181.51 | 205.62 |
| MAE ($), zero prediction | 145.66 | 175.66 | 199.93 |
| skill vs zero (1 - MSE / MSE of 0), served | -0.0001 | n/a (beta = 0: served delta is 0) | -0.0010 |
| skill vs zero, raw heads | -0.0106 | -0.0616 | -0.0496 |
| EV, served | 0.0000 | n/a (beta = 0: served delta is 0) | -0.0010 |
| EV, raw heads | -0.0092 | -0.0608 | -0.0497 |
| corr, Pearson, raw heads (the same for served while beta > 0) | 0.0064 | -0.0453 | -0.0283 |
| corr, Spearman, raw heads | -0.0037 | -0.0301 | -0.0236 |
| mean predicted ($), served | -0.41 | 0.00 | -0.01 |
| mean predicted ($), raw heads | -3.36 | -2.39 | -0.16 |
| mean realised ($) | 6.30 | 9.37 | 12.34 |
| share predicted up, raw heads | 0.3910 | 0.4977 | 0.5482 |
| share realised up | 0.5135 | 0.5146 | 0.5156 |
| shrink beta (served = beta x raw, fit on cal) | 0.1234 | 0.0000 | 0.0695 |

Correlations and the share predicted up are the raw heads': the served delta, beta x raw, has the same while beta > 0. beta = 0 for h1: the served delta is 0 there, the zero prediction (its errors are the zero prediction's, its skill vs zero and EV 0 by construction: n/a), so it has no correlation and no sign of its own.

## Variance heads

| metric | h0 (10 bars) | h1 (15 bars) | h2 (20 bars) |
|---|---|---|---|
| CRPS ($) | 105.50 | 127.33 | 145.01 |
| CRPSS vs constant variance | 0.0220 | 0.0167 | 0.0193 |
| NLL | 6.6473 | 6.8441 | 6.9669 |
| PIT KS | 0.0348 | 0.0328 | 0.0340 |
| var / err^2 Spearman | 0.2498 | 0.2243 | 0.2275 |
| coverage of the 90% interval | 0.9035 | 0.9059 | 0.9116 |
| width of the 90% interval ($) | 645.00 | 781.37 | 911.32 |

## Confidence gap (accuracy of the more confident half minus the less confident half)

| horizon | gap | 95% CI | verdict |
|---|---|---|---|
| h0 | 0.0124 | [-0.0281, 0.0505] | NOISE |
| h1 | -0.0188 | [-0.0542, 0.0202] | NOISE |
| h2 | 0.0020 | [-0.0418, 0.0479] | NOISE |

## Coherence across horizons

Magnitude ordering: the share of samples whose predicted move grows with the horizon, as the loss asks. Magnitudes in random order would give 0.5, 0.5 and 0.1667.

| check | raw price heads (the trained ordering) | served deltas (beta-shrunk: h0 0.123 / h1 0.000 / h2 0.069) | realised moves |
|---|---|---|---|
| abs(d h0) <= abs(d h1) | 0.7919 | n/a (beta = 0: served delta is 0) | 0.6039 |
| abs(d h1) <= abs(d h2) | 0.7974 | n/a (beta = 0: served delta is 0) | 0.5712 |
| full chain h0 <= h1 <= h2 | 0.6176 | n/a (beta = 0: served delta is 0) | 0.3159 |

beta = 0 for h1: the served delta is 0 there, so it has no magnitude ordering (|0| <= |0| holds on every bar by ties) and no sign; the served checks that involve it are n/a; the sign checks below use the raw heads.

The served ordering mostly reflects the per-horizon shrink beta, not the model: judge the trained constraint on the raw heads.

Sign agreement: sign(raw price head) against calibrated P(up) > 0.5 (the served delta, beta x raw, has the same sign while beta > 0).

| | h0 | h1 | h2 | all 3 |
|---|---|---|---|---|
| agree | 0.5703 | 0.5239 | 0.5813 | 0.1625 |
| expected if the two signs were independent | 0.4997 | 0.5003 | 0.4975 | 0.1127 |

- P(up) unanimity (all three horizons call the same side): 0.2258

## Against baselines (fit on the training block)

Each cell: model vs baseline, the margin (positive = the model is better) and the verdict. "noise": |z| < 1.96, so the ordering is not established; "significantly worse": the model loses with z <= -1.96. "DM z": Diebold-Mariano test of the per-sample loss difference (RMSE and skill, MAE, Brier, accuracy, CRPS, NLL) with a Bartlett (Newey-West) long-run variance, lag 2 x bars ahead. "boot z": the margin over its standard error in a paired moving-block bootstrap (80-bar blocks, 500 resamples; MCC, AUC, balanced accuracy, ECE, EV, corr, PIT KS, var / err^2 Spearman). Rows that only restate the RMSE verdict for a constant prediction (EV, corr and skill against zero_delta / mean_delta) are left out; the JSON keeps every verdict.

| baseline | metric | h0 (10 bars) | h1 (15 bars) | h2 (20 bars) |
|---|---|---|---|---|
| logreg_lags | direction/mcc | -0.0299 vs 0.0390 (-0.0689): does not beat, noise (boot z -1.72) | -0.0340 vs 0.0397 (-0.0737): does not beat, significantly worse (boot z -2.41) | 0.0063 vs 0.0411 (-0.0349): does not beat, noise (boot z -0.81) |
| logreg_lags | direction/auc | 0.4855 vs 0.5241 (-0.0386): does not beat, noise (boot z -1.70) | 0.4781 vs 0.5228 (-0.0447): does not beat, significantly worse (boot z -2.31) | 0.5044 vs 0.5319 (-0.0275): does not beat, noise (boot z -1.07) |
| logreg_lags | direction/brier | 0.2519 vs 0.2495 (-0.0024): does not beat, noise (DM z -1.71) | 0.2529 vs 0.2493 (-0.0036): does not beat, significantly worse (DM z -3.16) | 0.2507 vs 0.2492 (-0.0015): does not beat, noise (DM z -1.12) |
| logreg_lags | direction/ece_pos | 0.0391 vs 0.0211 (-0.0180): does not beat, noise (boot z -1.10) | 0.0477 vs 0.0197 (-0.0280): does not beat, noise (boot z -1.75) | 0.0226 vs 0.0231 (+0.0005): beats, noise (boot z +0.04) |
| logreg_lags | direction/acc | 0.4849 vs 0.5160 (-0.0311): does not beat, noise (DM z -1.55) | 0.4811 vs 0.5166 (-0.0355): does not beat, significantly worse (DM z -2.34) | 0.5026 vs 0.5157 (-0.0131): does not beat, noise (DM z -0.62) |
| logreg_lags | direction/bal_acc | 0.4850 vs 0.5190 (-0.0340): does not beat, noise (boot z -1.72) | 0.4831 vs 0.5195 (-0.0363): does not beat, significantly worse (boot z -2.41) | 0.5031 vs 0.5200 (-0.0169): does not beat, noise (boot z -0.80) |
| class_prior | direction/mcc | -0.0299 vs 0.0000 (-0.0299): does not beat, noise (boot z -1.17) | -0.0340 vs 0.0000 (-0.0340): does not beat, noise (boot z -1.08) | 0.0063 vs 0.0000 (+0.0063): beats, noise (boot z +0.20) |
| class_prior | direction/auc | 0.4855 vs 0.5000 (-0.0145): does not beat, noise (boot z -0.92) | 0.4781 vs 0.5000 (-0.0219): does not beat, noise (boot z -1.13) | 0.5044 vs 0.5000 (+0.0044): beats, noise (boot z +0.22) |
| class_prior | direction/brier | 0.2519 vs 0.2502 (-0.0017): does not beat, noise (DM z -1.91) | 0.2529 vs 0.2502 (-0.0028): does not beat, significantly worse (DM z -2.29) | 0.2507 vs 0.2502 (-0.0005): does not beat, noise (DM z -0.39) |
| class_prior | direction/ece_pos | 0.0391 vs 0.0208 (-0.0182): does not beat, noise (boot z -1.07) | 0.0477 vs 0.0196 (-0.0281): does not beat, noise (boot z -1.57) | 0.0226 vs 0.0234 (+0.0008): beats, noise (boot z +0.06) |
| class_prior | direction/acc | 0.4849 vs 0.4861 (-0.0011): does not beat, noise (DM z -0.05) | 0.4811 vs 0.4853 (-0.0042): does not beat, noise (DM z -0.19) | 0.5026 vs 0.4811 (+0.0215): beats, noise (DM z +0.80) |
| class_prior | direction/bal_acc | 0.4850 vs 0.5000 (-0.0150): does not beat, noise (boot z -1.17) | 0.4831 vs 0.5000 (-0.0169): does not beat, noise (boot z -1.08) | 0.5031 vs 0.5000 (+0.0031): beats, noise (boot z +0.20) |
| zero_delta | delta/rmse | 196.23 vs 196.21 (-0.01, -0.01%): does not beat, noise (DM z -0.15) | 236.10 vs 236.10 (+0.00, +0.00%): does not beat | 269.23 vs 269.11 (-0.13, -0.05%): does not beat, noise (DM z -0.91) |
| zero_delta | delta/mae | 145.68 vs 145.66 (-0.02, -0.02%): does not beat, noise (DM z -0.38) | 175.66 vs 175.66 (+0.00, +0.00%): does not beat | 200.03 vs 199.93 (-0.10, -0.05%): does not beat, noise (DM z -0.94) |
| mean_delta | delta/rmse | 196.23 vs 196.24 (+0.01, +0.00%): beats, noise (DM z +0.11) | 236.10 vs 236.14 (+0.04, +0.02%): beats, noise (DM z +1.09) | 269.23 vs 269.17 (-0.07, -0.02%): does not beat, noise (DM z -0.45) |
| mean_delta | delta/mae | 145.68 vs 145.68 (-0.00, -0.00%): does not beat, noise (DM z -0.07) | 175.66 vs 175.69 (+0.03, +0.02%): beats, noise (DM z +0.94) | 200.03 vs 199.97 (-0.06, -0.03%): does not beat, noise (DM z -0.49) |
| const_var | variance/crps | 105.50 vs 107.87 (+2.37, +2.20%): beats (DM z +6.21) | 127.33 vs 129.50 (+2.16, +1.67%): beats (DM z +4.00) | 145.01 vs 147.86 (+2.85, +1.93%): beats (DM z +4.02) |
| const_var | variance/nll | 6.6473 vs 6.7057 (+0.0584): beats (DM z +3.95) | 6.8441 vs 6.8904 (+0.0463): beats (DM z +2.55) | 6.9669 vs 7.0219 (+0.0550): beats (DM z +2.70) |
| const_var | variance/pit_ks | 0.0348 vs 0.0663 (+0.0315): beats (boot z +5.66) | 0.0328 vs 0.0614 (+0.0286): beats (boot z +4.68) | 0.0340 vs 0.0644 (+0.0304): beats (boot z +4.89) |
| const_var | variance/corr_var_err2_spearman | 0.2498 vs 0.0000 (+0.2498): beats (boot z +6.96) | 0.2243 vs 0.0000 (+0.2243): beats (boot z +5.65) | 0.2275 vs 0.0000 (+0.2275): beats (boot z +5.23) |


### The numbers per horizon (test block)

Classification per horizon (outside the deadband), and the price heads' errors in dollars: raw, served (beta x raw) and predicting zero.

In [7]:
display(AT.styled(AT.classification_table(test, ctx.config)))
display(AT.styled(AT.delta_quality_table(test, ctx.config, raw_delta=raw_test, delta_scale=betas)))

horizon,h0,h1,h2
metric,,,
bars ahead,10,15,20
n samples,7236,7236,7236
n scored (moves beyond 5 bps),5345,5697,5874
n_eff of the scored moves (n scored // bars ahead),534,379,293
true up-rate,0.5139,0.5147,0.5189
calls up (predicted up-rate),0.4958,0.4308,0.4848
accuracy,0.4849,0.4811,0.5026
accuracy 95% CI (n_eff),"[0.4428, 0.5273]","[0.4313, 0.5313]","[0.4457, 0.5593]"
majority-class accuracy (hindsight),0.5139,0.5147,0.5189


horizon,h0,h1,h2
metric,,,
bars ahead,10,15,20
n samples,7236,7236,7236
n_eff (non-overlapping outcomes),723,482,361
shrink beta (served = beta x raw),0.1234,0.0000,0.0695
"RMSE ($), raw heads",197.25,243.27,275.70
"RMSE ($), served",196.23,236.10,269.23
"RMSE ($), zero prediction",196.21,236.10,269.11
"MAE ($), raw heads",146.68,181.51,205.62
"MAE ($), served",145.68,175.66,200.03


## Model analytics on the TEST block

**Direction heads**: does the served P(up) differ between bars that went up and bars that went down? Row 2 draws
the ROC as lift over chance (TPR - FPR; the area is AUC - 0.5) for the direction head and the price head's
Gaussian readout, inside the band a no-skill head stays in. Row 3 shows the reliability of raw (open) vs
calibrated (filled) P(up) in 10 equal-count bins with block-clustered 95% intervals. The scorecard repeats the
evaluation report's numbers against a constant 0.5.

In [8]:
Visualizations.build("direction_analytics", test, ctx.config, raw_delta=raw_test).show()

**Price heads**: the per-horizon error table (raw vs served vs predicting 0, with intervals), every sample's predicted vs realised move, the binned calibration curve, and how the correlation and the served skill drift along the block against their no-skill bands.

In [9]:
Visualizations.build("delta_analytics", test, ctx.config, raw_delta=raw_test).show()

**Variance heads**: does the predicted sigma size and rank the realised error, against a free baseline (the trailing 60-bar realised vol the conformal intervals use)? Is the error shape Gaussian (PIT, tail rates)? Do the 90% intervals hold their coverage and width along the block?

In [10]:
Visualizations.build("variance_analytics", test, ctx.config, raw_delta=raw_test).show()

**Direction confidence.** Row 1: accuracy by decile of |P(up) - 0.5| with 95% block-bootstrap intervals, the
accuracy a calibrated P(up) would reach (diamonds) and the no-skill level of each decile's call mix (ticks).
Row 2: selective accuracy, keeping only the most decided x%. Row 3: the same for the strategies' confidence
exp(-var / var_scale), which gates and sizes their trades. Row 4: confusion matrices with recall, precision,
balanced accuracy and MCC.

In [11]:
Visualizations.build("confidence_analytics", test, ctx.config, var_scale=var_scale_from(cal), report=report).show()

**Cross-horizon coherence.** P(up) correlation between horizons; the eight up/down vote patterns against what
independent votes would give; the realised up-rate by the number of horizons voting up; direction-head vs
price-head sign agreement; whether |delta| grows with the horizon on the raw heads vs the served (shrunk) deltas;
and the strategies' vote agreement at their vote lines.

In [12]:
Visualizations.build("coherence_analytics", test, ctx.config, raw_delta=raw_test).show()
display(AT.styled(AT.magnitude_ordering_table(test, raw_delta=raw_test)))
display(AT.styled(AT.alignment_table(test, ctx.config, raw_delta=raw_test)))

,raw heads (trained ordering),raw heads 95% CI (n_eff),served deltas,served deltas 95% CI (n_eff),realised moves,realised moves 95% CI (n_eff),magnitudes in random order,n_eff
check,,,,,,,,
|d h0| <= |d h1|,0.7919,"[0.7534, 0.8257]",n/a,n/a (beta = 0: served delta is 0),0.6039,"[0.5596, 0.6466]",0.5000,482
|d h1| <= |d h2|,0.7974,"[0.7530, 0.8356]",n/a,n/a (beta = 0: served delta is 0),0.5712,"[0.5197, 0.6212]",0.5000,361
full chain |d h0| <= |d h1| <= |d h2|,0.6176,"[0.5665, 0.6662]",n/a,n/a (beta = 0: served delta is 0),0.3159,"[0.2702, 0.3655]",0.1667,361


,agree,95% CI low,95% CI high,expected if independent,share delta > 0,share P(up) > 0.5,n,n_eff
horizon,,,,,,,,
h0,0.5703,0.5340,0.6059,0.4997,0.3910,0.5014,7236,723
h1,0.5239,0.4793,0.5681,0.5003,0.4977,0.4323,7236,482
h2,0.5813,0.5298,0.6310,0.4975,0.5482,0.4736,7236,361
all 3,0.1625,0.1281,0.2040,0.1127,n/a,n/a,7236,361


Trailing move (the extended-trend feature and the momentum prior) vs the realised move; a correlation whose sign changes between blocks is not a stable edge.

In [13]:
display(AT.styled(AT.trailing_move_table(
    {"cal": cal, "test": test}, ctx.config,
    raw_deltas={"cal": result.predictions_cal["delta"], "test": raw_test},
    trends={"cal": blocks["cal"]["extended_trends"], "test": blocks["test"]["extended_trends"]})))

## Learned indicator periods

The lines are the base periods per epoch; the model applies a per-window shifted period, whose medians on the
test windows (served weights) are the diamonds right of the last epoch. Change % is measured from the configured
start. The correlation bars are epoch-to-epoch changes against a noise band.

In [14]:
metrics_path = ctx.path("metrics.jsonl")
applied = applied_periods(result, result.windows_test, block="test")
Visualizations.build("indicator_evolution", metrics_path, ctx.config, applied=applied).show()
indicator_applied_periods(applied, ctx.config, metrics=metrics_path).show()
display(AT.styled(indicator_summary(metrics_path, ctx.config, applied=applied)))

,indicator,start,after epoch 1,last,min,max,change %,slope %/epoch (last 5),headroom %,near bound,CV,corr with val loss,r with epoch,r of changes,served base,applied p5,applied p50,applied p95,applied p50 vs base %
macd_1_slow,MACD #1 slow,35.0000,35.9315,59.2843,35.0000,60.0000,69.3839,-0.2956,1.2072,ceiling,0.1489,-0.9352,0.9583,0.0959,58.9089,53.3215,73.0178,93.7891,23.9504
macd_2_fast,MACD #2 fast,8.0000,8.3440,4.3730,4.3730,8.3440,-45.3374,-0.8394,118.6502,,0.2077,0.9744,-0.9314,0.5030,4.5141,3.5015,3.9795,4.3026,-11.8414
macd_0_fast,MACD #0 fast,12.0000,11.6427,6.5878,6.5878,12.0000,-45.1017,-0.7535,229.3896,,0.1647,0.9527,-0.9320,0.3376,6.6691,4.5210,5.3324,6.6681,-20.0425
ma_period_2,MA #2,30.0000,26.9815,18.7817,18.7817,30.0000,-37.3944,-0.3224,219.4603,,0.1068,0.9555,-0.9473,0.2764,18.8929,17.2226,22.9392,29.6204,21.4169
macd_2_slow,MACD #2 slow,17.0000,15.7236,22.2916,15.7236,22.3066,31.1272,1.2503,169.1593,,0.0941,-0.7978,0.8902,0.1569,22.3066,16.8794,21.6415,27.0633,-2.9818
macd_0_slow,MACD #0 slow,26.0000,25.9352,33.1523,25.9352,33.1523,27.5087,1.1696,80.9831,,0.0559,-0.7194,0.8342,0.3349,32.3222,22.2466,26.3788,29.8235,-18.3880
rsi_period_0,RSI #0,9.0000,8.8537,6.7362,6.6176,9.0000,-25.1528,-1.1739,236.8122,,0.0920,0.9552,-0.9733,0.4255,6.6559,6.8990,7.6992,9.2882,15.6745
ma_period_1,MA #1,10.0000,9.4482,7.5908,7.2301,10.0000,-24.0922,1.1236,279.5390,,0.0880,0.9362,-0.9555,0.1514,7.3659,7.4808,8.2536,9.7418,12.0515
ma_period_0,MA #0,5.0000,4.7468,3.9455,3.9455,5.0000,-21.0908,-1.3968,97.2730,,0.0576,0.6180,-0.8217,-0.1719,4.0731,3.1907,3.9337,4.9629,-3.4238
macd_1_fast,MACD #1 fast,5.0000,5.0893,3.9732,3.8031,5.1663,-20.5366,0.0876,98.6585,,0.1019,0.9212,-0.8924,0.2111,4.0663,2.8947,3.4989,4.8136,-13.9529
